# Notebook 02 — Exploratory Analysis & Visualisation
**Authors:** Adeesha & Thaweesha  
**Role:** Documentation & Visualisation  
**Task:** Load cleaned data from HDFS → Analyse → Create charts → Save outputs locally

---
> **Prerequisite:** `01_data_cleaning.ipynb` must have been run first.

## 1. Load Packages

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Output directory for charts and CSVs
os.makedirs("output/visualisations", exist_ok=True)

# Chart style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"

print("Packages loaded.")

## 2. Start Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("RetailVisualisation") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 3. Load Cleaned Dataset from Hadoop HDFS

In [ ]:
# Load the cleaned Parquet from HDFS — saved by Notebook 01
df = spark.read.parquet("hdfs:///retail/cleaned/")

print(f"Loaded {df.count():,} records from HDFS.")
print(f"Columns: {df.columns}")
df.show(3, truncate=True)

## 4. Dataset Overview

In [ ]:
# High-level statistics
total_revenue   = df.agg(F.round(F.sum("TotalPrice"), 2).alias("TotalRevenue")).collect()[0][0]
total_orders    = df.select("Invoice").distinct().count()
total_customers = df.select("CustomerID").distinct().count()
total_products  = df.select("StockCode").distinct().count()
total_countries = df.select("Country").distinct().count()

print("=" * 42)
print(f"  Total Revenue    : £{total_revenue:>12,.2f}")
print(f"  Total Orders     : {total_orders:>12,}")
print(f"  Total Customers  : {total_customers:>12,}")
print(f"  Unique Products  : {total_products:>12,}")
print(f"  Countries        : {total_countries:>12,}")
print("=" * 42)

## 5. Chart 1 — Top 10 Countries by Revenue
**Adeesha**

In [ ]:
# Aggregate revenue by country
country_rev = df.groupBy("Country") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("Revenue")) \
    .orderBy(F.desc("Revenue")) \
    .limit(10) \
    .toPandas()

# Save aggregated result locally
country_rev.to_csv("output/visualisations/top10_countries_revenue.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#1F4E79" if i == 0 else "#2E75B6" for i in range(len(country_rev))]
bars = ax.barh(country_rev["Country"][::-1], country_rev["Revenue"][::-1], color=colors[::-1])

ax.set_xlabel("Total Revenue (£)", fontsize=12)
ax.set_title("Top 10 Countries by Revenue", fontsize=14, fontweight="bold", pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e6:.1f}M"))

# Add value labels
for bar in bars:
    w = bar.get_width()
    ax.text(w + w*0.01, bar.get_y() + bar.get_height()/2,
            f"£{w/1e6:.2f}M", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("output/visualisations/chart1_top10_countries.png", bbox_inches="tight")
plt.show()
print("Chart 1 saved.")

**Interpretation:** The United Kingdom dominates revenue by a large margin, as expected for a UK-based retailer. The next highest countries — Netherlands, EIRE, and Germany — each contribute significantly less, highlighting the retailer's strong domestic market and modest international presence.

## 6. Chart 2 — Monthly Revenue Trend
**Adeesha**

In [ ]:
# Monthly revenue aggregation
monthly_rev = df.groupBy("YearMonth") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("Revenue"),
         F.countDistinct("Invoice").alias("OrderCount")) \
    .orderBy("YearMonth") \
    .toPandas()

monthly_rev.to_csv("output/visualisations/monthly_revenue.csv", index=False)

# Plot
fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.plot(monthly_rev["YearMonth"], monthly_rev["Revenue"],
         color="#1F4E79", marker="o", linewidth=2, markersize=5, label="Revenue")
ax1.fill_between(monthly_rev["YearMonth"], monthly_rev["Revenue"],
                 alpha=0.15, color="#2E75B6")
ax1.set_ylabel("Revenue (£)", fontsize=12, color="#1F4E79")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e3:.0f}K"))

ax2 = ax1.twinx()
ax2.bar(monthly_rev["YearMonth"], monthly_rev["OrderCount"],
        alpha=0.25, color="#E07B39", label="Orders")
ax2.set_ylabel("Number of Orders", fontsize=12, color="#E07B39")

ax1.set_xlabel("Month", fontsize=12)
ax1.set_title("Monthly Revenue & Order Count Trend", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.savefig("output/visualisations/chart2_monthly_revenue.png", bbox_inches="tight")
plt.show()
print("Chart 2 saved.")

**Interpretation:** Revenue shows clear seasonal patterns with a strong peak in November — driven by Christmas shopping. There is a consistent upward trend across 2010 compared to 2009, suggesting business growth. The lowest months are typically January and February, reflecting post-holiday slowdown.

## 7. Chart 3 — Top 10 Best-Selling Products
**Adeesha**

In [ ]:
# Top 10 products by total quantity sold
top_products = df.groupBy("StockCode", "Description") \
    .agg(F.sum("Quantity").alias("TotalQty"),
         F.round(F.sum("TotalPrice"), 2).alias("TotalRevenue")) \
    .orderBy(F.desc("TotalQty")) \
    .limit(10) \
    .toPandas()

# Shorten long descriptions for display
top_products["ShortDesc"] = top_products["Description"].str[:30]
top_products.to_csv("output/visualisations/top10_products.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(11, 6))
palette = sns.color_palette("Blues_r", len(top_products))
bars = ax.barh(top_products["ShortDesc"][::-1], top_products["TotalQty"][::-1], color=palette)

ax.set_xlabel("Total Quantity Sold", fontsize=12)
ax.set_title("Top 10 Best-Selling Products by Quantity", fontsize=14, fontweight="bold")

for bar in bars:
    w = bar.get_width()
    ax.text(w + w*0.005, bar.get_y() + bar.get_height()/2,
            f"{int(w):,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("output/visualisations/chart3_top10_products.png", bbox_inches="tight")
plt.show()
print("Chart 3 saved.")

**Interpretation:** The top-selling products are mostly small, low-cost decorative or gift items sold in high volumes. These fast-moving items drive order frequency and are strong candidates for cross-sell and bundle promotions.

## 8. Chart 4 — Orders by Day of Week
**Thaweesha**

In [ ]:
# Day names for mapping (Spark: 1=Sunday, 2=Monday, ..., 7=Saturday)
day_map = {1: "Sunday", 2: "Monday", 3: "Tuesday", 4: "Wednesday",
           5: "Thursday", 6: "Friday", 7: "Saturday"}

orders_by_day = df.groupBy("DayOfWeek") \
    .agg(F.countDistinct("Invoice").alias("OrderCount"),
         F.round(F.sum("TotalPrice"), 2).alias("Revenue")) \
    .orderBy("DayOfWeek") \
    .toPandas()

orders_by_day["DayName"] = orders_by_day["DayOfWeek"].map(day_map)
orders_by_day.to_csv("output/visualisations/orders_by_day.csv", index=False)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Orders
axes[0].bar(orders_by_day["DayName"], orders_by_day["OrderCount"],
            color=["#1F4E79" if d not in ["Saturday", "Sunday"] else "#AACFE4"
                   for d in orders_by_day["DayName"]])
axes[0].set_title("Orders by Day of Week", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Orders")
axes[0].set_xticklabels(orders_by_day["DayName"], rotation=30, ha="right")

# Revenue
axes[1].bar(orders_by_day["DayName"], orders_by_day["Revenue"],
            color=["#2E75B6" if d not in ["Saturday", "Sunday"] else "#AACFE4"
                   for d in orders_by_day["DayName"]])
axes[1].set_title("Revenue by Day of Week", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Revenue (£)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e6:.1f}M"))
axes[1].set_xticklabels(orders_by_day["DayName"], rotation=30, ha="right")

plt.suptitle("Trading Patterns by Day", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("output/visualisations/chart4_orders_by_day.png", bbox_inches="tight")
plt.show()
print("Chart 4 saved.")

**Interpretation:** Thursday and Wednesday are the busiest trading days for both orders and revenue. Sunday has almost no activity, and Saturday is also very low — suggesting this is a B2B (business-to-business) retailer where customers are businesses that order during the working week.

## 9. Chart 5 — Revenue Distribution per Order
**Thaweesha**

In [ ]:
# Order-level revenue (sum per invoice)
order_rev = df.groupBy("Invoice") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("OrderRevenue")) \
    .filter(F.col("OrderRevenue") < 5000)  # Remove extreme outliers for visualisation

order_rev_pd = order_rev.toPandas()
order_rev_pd.to_csv("output/visualisations/order_revenue_distribution.csv", index=False)

# Plot histogram
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(order_rev_pd["OrderRevenue"], bins=80, color="#2E75B6", edgecolor="white", alpha=0.85)
ax.axvline(order_rev_pd["OrderRevenue"].median(), color="#E07B39",
           linestyle="--", linewidth=2, label=f'Median: £{order_rev_pd["OrderRevenue"].median():.0f}')
ax.axvline(order_rev_pd["OrderRevenue"].mean(), color="#C00000",
           linestyle="-", linewidth=2, label=f'Mean: £{order_rev_pd["OrderRevenue"].mean():.0f}')

ax.set_xlabel("Order Revenue (£)", fontsize=12)
ax.set_ylabel("Number of Orders", fontsize=12)
ax.set_title("Revenue Distribution per Order (Orders < £5,000)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("output/visualisations/chart5_order_distribution.png", bbox_inches="tight")
plt.show()
print("Chart 5 saved.")

**Interpretation:** The distribution is heavily right-skewed — the vast majority of orders are under £500, but a small number of very large bulk orders exist. The median order value is much lower than the mean, confirming this skew. This is typical for a wholesale/gift retailer.

## 10. Chart 6 — Top 10 Customers by Total Spend
**Thaweesha**

In [ ]:
# Top customers by total spend
top_customers = df.groupBy("CustomerID") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("TotalSpend"),
         F.countDistinct("Invoice").alias("OrderCount")) \
    .orderBy(F.desc("TotalSpend")) \
    .limit(10) \
    .toPandas()

top_customers["CustomerLabel"] = "Cust " + top_customers["CustomerID"].astype(str)
top_customers.to_csv("output/visualisations/top10_customers.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(top_customers["CustomerLabel"], top_customers["TotalSpend"],
              color=sns.color_palette("Blues_r", 10))

ax.set_xlabel("Customer ID", fontsize=12)
ax.set_ylabel("Total Spend (£)", fontsize=12)
ax.set_title("Top 10 Customers by Total Spend", fontsize=14, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"£{x/1e3:.0f}K"))

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + h*0.01,
            f"£{h/1e3:.1f}K", ha="center", va="bottom", fontsize=9)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("output/visualisations/chart6_top_customers.png", bbox_inches="tight")
plt.show()
print("Chart 6 saved.")

**Interpretation:** The top customer spends significantly more than the rest, suggesting a small number of high-value wholesale buyers who are critical to business revenue. These customers should be prioritised for loyalty and retention strategies.

## 11. Chart 7 — Revenue Heatmap: Month vs Day of Week
**Thaweesha**

In [ ]:
day_map = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}
month_map = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
             7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}

heatmap_data = df.groupBy("Month", "DayOfWeek") \
    .agg(F.round(F.sum("TotalPrice"), 2).alias("Revenue")) \
    .toPandas()

heatmap_data["DayName"]   = heatmap_data["DayOfWeek"].map(day_map)
heatmap_data["MonthName"] = heatmap_data["Month"].map(month_map)

pivot = heatmap_data.pivot(index="DayName", columns="MonthName", values="Revenue")
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
day_order   = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
pivot = pivot.reindex(index=day_order, columns=[m for m in month_order if m in pivot.columns])

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(pivot / 1e3, cmap="Blues", annot=True, fmt=".0f",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Revenue (£K)"})
ax.set_title("Revenue Heatmap — Month vs Day of Week (£K)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Day of Week")

plt.tight_layout()
plt.savefig("output/visualisations/chart7_heatmap.png", bbox_inches="tight")
plt.show()
print("Chart 7 saved.")

**Interpretation:** The heatmap confirms Thursday–Friday in October and November are the highest-revenue combinations, strongly driven by holiday buying. Weekends and summer months (June–August) consistently show lower values.

## 12. Save All Visualisation Results

In [ ]:
# Save a combined summary CSV for the report
summary = pd.DataFrame({
    "Metric": ["Total Revenue", "Total Orders", "Total Customers",
               "Unique Products", "Countries Served"],
    "Value":  [f"£{total_revenue:,.2f}", f"{total_orders:,}",
               f"{total_customers:,}", f"{total_products:,}", f"{total_countries:,}"]
})
summary.to_csv("output/visualisations/dataset_summary.csv", index=False)

print("All outputs saved to: output/visualisations/")
print("Files saved:")
for f in sorted(os.listdir("output/visualisations")):
    print(f"  {f}")

---
## Summary

| Chart | What It Shows | Key Finding |
|-------|--------------|-------------|
| 1. Top Countries | Revenue by country | UK dominates overwhelmingly |
| 2. Monthly Trend | Revenue over time | Peak in Nov, growth year-on-year |
| 3. Top Products | Best sellers by quantity | Small decorative items sell most |
| 4. Day of Week | When orders happen | Midweek (Thu) busiest |
| 5. Order Distribution | Spread of order values | Most orders are under £500 |
| 6. Top Customers | Highest spenders | Small group of wholesale buyers |
| 7. Heatmap | Month × Day revenue | Nov Thu/Fri = peak |

**Next:** Run `03_frequently_bought.ipynb`